# ECON2041 Week 5 tutorial: samples, estimators, and how variables move together

### What you'll be able to do by the end

- Tell a sample from a population, and a statistic from a parameter
- Watch an estimate change from one random sample to the next, and describe the spread of estimates you could have got
- Compute a covariance and a correlation, and say why correlation is easier to compare across variables or datasets
- Compute a conditional expectation and read it as a function of the variable we condition on

### Different types of cells

- 🟢 Read and run: these cells have pre-written code, you can simply run them, read and interpret the output
- ✏️ You write: your turn to try to write code or sometimes add a short written answer
- 🤔 You think: stop and think on your own before running the next cell or reading on
- 💬 Discuss: talk with your neighbors before typing or running

## 0. Setup

The same setup block as last week.
As always, we run it at the start of a session.

In [ ]:
# 🟢 Read and run: our standard ECON2041 setup block
import numpy as np               # numerical tools (nicknamed np)
import pandas as pd              # data tools (nicknamed pd)
import matplotlib.pyplot as plt  # plotting tools (nicknamed plt)
import seaborn as sns            # statistical charts (nicknamed sns)

# Keep scalar output plain under NumPy 2 (0.5, not np.float64(0.5)).
if np.lib.NumpyVersion(np.__version__) >= "2.0.0":
    np.set_printoptions(legacy="1.25")

DATA = "https://emiliatjernstrom.com/econ2041/data"   # Unit datasets live at this web address

print("Setup done!")

## 1. Populations, samples, statistics, parameters

### Question 1: a number from the news

The week 4 Essential Concepts recordings separated 
- a **population** (all the units we care about) from 
- a **sample** (the subset we actually observe), and 
- a **parameter** (a fixed feature of the population, written with Greek letters like $\mu$) from 
- a **statistic** (a number computed from a sample, written with Roman letters like $\bar{x}$)

💬 With the person next to you, choose one of these three headlines:

- [Unemployment rate jumps to 4.5 per cent, driven by female unemployment](https://www.abc.net.au/news/2026-05-21/unemployment-rate-jumps-4-5-per-cent/106705576)
- [Few Australians feel financially better off, new poll shows](https://www.abc.net.au/news/2025-03-31/cost-of-living-yougov-polling-election-campaign-budget-reaction/105063404)
- ['Dwindling' affordability: Sydney's median house price hits record $1.76m](https://www.domain.com.au/news/dwindling-affordability-sydneys-median-house-price-hits-record-1-76m-2-1475446/)

For your headline, decide:

- What population does the headline claim or imply that it describes?
- Who or what was actually observed to produce the number?
- Is the number a parameter or a statistic? If the answer depends on how you define the population, explain why

### The dataset: `pisa-australia-2022.csv`

Same file as last week: 
- Australian students in PISA 2022
- one row per student, and for each student, we know
  - `math`, 
  - `read`, and 
  - `science` scores, 
  - the number of books at home (`book`), 
  - the index of economic, social and cultural status (`escs`), 
  - and `gender`

Source: [OECD PISA](https://www.oecd.org/en/about/programmes/pisa.html), via the [learningtower project](https://kevinwang09.github.io/learningtower/)


- In 2022, PISA tested a sample of 13,437 Australian students
- 12,136 of the tested students have complete records in our file (recall that we drop some observations who have missing values for some of our variables)

In reality, these students are a **sample** of Australian 15-year-olds. 

For today's exercise, however, we will treat the 12,136 students in our file as the **population**. 

This lets us pretend that we know the population values and watch what happens when we repeatedly draw different-sized samples. From this point onward, **population** means those 12,136 students.

In [ ]:
# 🟢 Read and run: load pisa-australia-2022.csv into a dataframe called pisa
pisa = pd.read_csv(f"{DATA}/pisa-australia-2022.csv")

# Store book as an ordered category so tables and charts use the natural order
book_order = ["1: 0-10", "2: 11-25", "3: 26-100", "4: 101-200", "5: 201-500", "6: 500+"]
pisa["book"] = pd.Categorical(pisa["book"], categories=book_order, ordered=True)

pisa.head()

### Question 2: the population mean math score

The week 4 Essential Concepts recordings write the population mean as $\mu$.
Because we are treating all 12,136 students in the file as our population, we can compute $\mu$ exactly. This population mean is the value our smaller samples will be trying to estimate.

In [ ]:
# 🟢 Read and run: the population mean math score
math_mean = pisa["math"].mean()
math_mean

## 2. Estimators are random variables

Suppose we could afford to survey only 200 of these 12,136 students, chosen at random.

> How far would the average of those 200 land from the population mean of 486.8?

We can find out because we know the population mean: draw 200 students at random, compute their mean, and compare.

### Question 3: one random sample of 200

In [ ]:
# 🟢 Read and run: draw 200 students at random and store them in a dataframe called sample_1
# random_state=1 fixes the "randomness", so everyone in the room draws the same 200 students
sample_1 = pisa.sample(200, random_state=1)

sample_1.head()

In [ ]:
# ✏️ You write: the average math score in sample_1
# Hint: same command as in question 2, asked of sample_1 instead of pisa

🤔 Change `random_state=1` to `random_state=2` in the cell that draws `sample_1`, and rerun both cells

- Did the estimate change?
- What about the estimator, i.e. the rule we applied?

(Then change it back to `random_state=1` and rerun, so that `sample_1` is the same 200 students for everyone in the next question)

### Question 4: a gap between two groups

The sample mean is the simplest statistic there is, so let's try one that answers a question we might actually ask.

> Do boys and girls in our population score differently in math?

Because we observe our entire population of 12,136 students, we can compute the mean math score of each gender and the gap between them. We can then see what a sample of 200 would have said.

In [ ]:
# 🟢 Read and run: the population mean math score by gender
gender_means = pisa.groupby("gender")["math"].mean()
gender_gap = gender_means["male"] - gender_means["female"]
gender_means

In [ ]:
# 🟢 Read and run: the distribution of math score, conditional on gender
plt.figure(figsize=(6, 3.5))
sns.histplot(
    data=pisa, x="math", hue="gender", bins=25,
    stat="probability", common_norm=False,
    multiple="layer", alpha=0.35
)
plt.xlabel("Math score")
plt.ylabel("Share of students")
plt.show()

🤔 Compare the centers, spread, and overlap of the two conditional distributions.
Does a 13.2-point difference in the means imply that boys' and girls' scores form two separate groups?

In [ ]:
# 🟢 Read and run: the same two means in sample_1
sample_1_gender_means = sample_1.groupby("gender")["math"].mean()
sample_1_gender_gap = sample_1_gender_means["male"] - sample_1_gender_means["female"]
sample_1_gender_means

💬 Suppose `sample_1` were the only data we observed.

- What boys-minus-girls gap would we report?
- Can this one sample tell us whether another random sample would give the same sign?
- What would we need to study to answer that second question?

### Question 5: 1,000 samples, and the spread of estimates

One sample tells us one estimate.
To see how much the estimate moves from sample to sample, we do the whole thing 1,000 times: draw 50 students, compute their mean, write it down, repeat.

> If we drew a random sample of 50 over and over, how would the sample means be spread around the population mean?

In [ ]:
# 🟢 Read and run: draw 1,000 samples of n students, and store each sample's mean math score
n = 50

means = []                                     # an empty list, to collect one mean per sample
for i in range(1000):                          # repeat 1,000 times, with i = 0, 1, 2, ..., 999
    draw = pisa.sample(n, random_state=i)      # a fresh random sample of n students
    means.append(draw["math"].mean())          # add this sample's mean to the list

print("first five sample means:", np.round(means[:5], 1))

In [ ]:
# 🟢 Read and run: the 1,000 sample means, with the population mean marked
plt.figure(figsize=(6, 3.5))
sns.histplot(means, bins=30, edgecolor="black", linewidth=0.7)
plt.axvline(math_mean, color="green", linewidth=2)   # the population mean
plt.xlim(430, 545)
plt.xlabel(f"Sample mean math score (n = {n})")
plt.ylabel("Number of samples")
plt.show()

🤔 Look at the histogram

- Where is the center of the histogram, compared with the green line?
- Roughly how far from the green line does a typical sample land?
- What is the worst sample among the 1,000, i.e. the one furthest from the population mean?

### Question 6: tutor demo - what changes with the sample size?

Watch as your tutor changes `n = 50` to `n = 200`, then to `n = 800`, and reruns the simulation and histogram.
The x-axis stays fixed, so the three histograms are directly comparable.

- As $n$ grows, what happens to the center?
- As $n$ grows, what happens to the spread?

### Question 7: tutor demo - back to the gap

In question 4, one sample of 200 got the sign of the boys-girls gap wrong.

> How often does a random sample get the sign wrong?

Watch as your tutor runs the same loop for the boys-minus-girls gap in mean math score.
We use `n = 50`, roughly the size of a couple of school classes.

In [ ]:
# 🟢 Tutor demo: 1,000 samples of 50 students, and each sample's boys-minus-girls gap in mean math
n_gap = 50

gaps = []
for i in range(1000):
    draw = pisa.sample(n_gap, random_state=i)
    means_by_gender = draw.groupby("gender")["math"].mean()
    gaps.append(means_by_gender["male"] - means_by_gender["female"])

sns.histplot(gaps, bins=30, edgecolor="black", linewidth=0.7)
plt.axvline(gender_gap, color="green", linewidth=2)   # the population gap, boys minus girls
plt.axvline(0, color="black", linestyle="--")   # no gap at all
plt.xlabel(f"Boys minus girls, mean math score (n = {n_gap})")
plt.ylabel("Number of samples")
plt.show()

In [ ]:
# 🟢 Tutor demo: the share of samples in which girls came out ahead
(np.array(gaps) < 0).mean()

💬 With 50 students, about what share of samples get the sign of the gap wrong?
If one sample reported that girls outscore boys in math by 20 points, how seriously would you take it?

## 3. Covariance and correlation, on real data

In the week 5 live lecture we took the covariance formula apart on 300 of these students, math score against family background.
Now we use the entire population of 12,136 students.

> Do students from more advantaged homes score higher in math?

### Question 8: the covariance, and its units problem

In [ ]:
# 🟢 Read and run: math score against the socio-economic index, one dot per student
sns.scatterplot(data=pisa, x="escs", y="math", s=8, alpha=0.2)
plt.xlabel("Economic, social and cultural status (escs)")
plt.ylabel("Math score")
plt.show()

🤔 Before computing anything: does the cloud lean up, lean down, or neither?

In [ ]:
# ✏️ You write: the covariance between math and escs
# Hint: pick out the two columns with a list, pisa[["math", "escs"]],
#       and call .cov() to get the covariance table

In [ ]:
# 🟢 Read and run: report math on a one-eighth scale
pisa["math_rescaled"] = pisa["math"] / 8

pisa[["math_rescaled", "escs"]].cov()

🤔 Same students, same relationship, and a different covariance. Which of the two numbers is "right"?

### Question 9: the correlation is unit-free

The week 4 Essential Concepts recordings define the correlation as the covariance divided by the two standard deviations:

$$\rho_{X,Y} = \frac{\text{cov}(X, Y)}{\sigma_X \, \sigma_Y}$$

Dividing by the standard deviations cancels the units, so the result is a number between $-1$ and $1$ that does not change under a positive linear rescaling.

🤔 Before computing it, guess the correlation, somewhere between $-1$ and $1$: ________

In [ ]:
# ✏️ You write: reuse your question 8 command, but ask for corr() instead of cov()

In [ ]:
# 🟢 Read and run: the correlation using the rescaled scores
pisa[["math_rescaled", "escs"]].corr()

🤔 The covariance changed when we rescaled the scores. Did the correlation?

## 4. The conditional expectation is a function

Section 3 summarized how `math` and `escs` move together in one number.
A conditional expectation keeps more: one expected value for every value of the conditioning variable.

> What math score should we expect from a student, given how many books they have at home?

### Question 10: the expected math score, given books at home

In [ ]:
# ✏️ You write: the mean math score within each book range
# Hint: adapt question 4: group by book, select math, call mean(), then round(1)

In [ ]:
# 🟢 Read and run: the six conditional means as a picture
# pointplot computes the mean of math within each book range and draws it as a dot
sns.pointplot(data=pisa, x="book", y="math", order=book_order, errorbar=None)
plt.xlabel("Books at home")
plt.ylabel("Mean math score, given books at home")
plt.show()

🤔 Read the picture

- What is the population value of $E[\text{math} \mid \text{book} = \text{4: 101-200}]$, roughly? How does it compare with the population mean of 486.8?
- Regression fits straight lines through clouds like the `math`-against-`escs` plot in section 3. Looking at these six dots, would a line summarize the upward pattern reasonably well?